In [ ]:
# --- ensure repo-root cwd ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# One vehicle — actual SoH + forecast to warranty

Actual SoH = coulomb-counting series (intellicar). Forecast = empirical capacity fade
`SoH = 100 − k·√t` fitted to this vehicle, extrapolated to the **battery warranty** horizon,
with a ±2σ(k) band and the 80% end-of-life line.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, DateFormatter

VIN = "MB7F8CLLFNJH48488"   # richest-history unit; change to any cohort VIN
WARRANTY_YEARS = 5.0         # assumption — change to your actual battery warranty
EOL = 80.0                   # end-of-life SoH threshold

m = pd.read_parquet("data/mahindra/features/feature_table.parquet")
g = m[m["vin"] == VIN].sort_values("month").copy()
t = g["age_months"].to_numpy(); s = g["soh"].to_numpy()
start = g["month"].min()
print(f"{VIN}: {len(g)} monthly SoH points, {start:%Y-%m} -> {g['month'].max():%Y-%m}, "
      f"current SoH {s[-1]:.1f}%")

In [ ]:
# Fit SoH = 100 - k*sqrt(t) through the origin (t in months); k and its std error.
mask = t > 0
x = np.sqrt(t[mask]); ymeas = 100.0 - s[mask]
k = np.sum(x * ymeas) / np.sum(x * x)
resid = ymeas - k * x
se_k = np.sqrt(np.sum(resid**2) / max(len(x) - 1, 1) / np.sum(x * x))

horizon = WARRANTY_YEARS * 12
tf = np.linspace(0, horizon, 200)
fit = 100 - k * np.sqrt(tf)
lo = 100 - (k + 2 * se_k) * np.sqrt(tf)
hi = 100 - (k - 2 * se_k) * np.sqrt(tf)
t_eol = ((100 - EOL) / k) ** 2 if k > 0 else np.inf
dates = start + pd.to_timedelta(tf * 30.4375, unit="D")
warranty_end = start + pd.DateOffset(years=int(WARRANTY_YEARS))
eol_date = start + pd.to_timedelta(t_eol * 30.4375, unit="D") if np.isfinite(t_eol) else None
soh_at_warranty = 100 - k * np.sqrt(horizon)
print(f"fade k = {k:.2f} ± {2*se_k:.2f} %/√month | SoH at {WARRANTY_YEARS:.0f}y = {soh_at_warranty:.1f}%")
print(f"reaches {EOL:.0f}% EOL at ~{t_eol:.0f} months ({eol_date:%Y-%m})" if eol_date is not None else "does not reach EOL")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
act_dates = start + pd.to_timedelta(t * 30.4375, unit="D")
ax.plot(act_dates, s, "o-", color="C0", lw=1.5, ms=5, label="actual SoH (coulomb counting)")
ax.plot(dates, fit, "--", color="C3", lw=2, label="forecast  SoH = 100 − k·√t")
ax.fill_between(dates, lo, hi, color="C3", alpha=0.15, label="±2σ band")
ax.axhline(EOL, ls=":", color="red", lw=1.5, label=f"{EOL:.0f}% end-of-life")
ax.axvline(warranty_end, ls="-.", color="green", lw=1.5, label=f"{WARRANTY_YEARS:.0f}-yr warranty")
ax.axvline(start + pd.DateOffset(years=3), ls="-.", color="grey", lw=1, alpha=0.6, label="3-yr mark")
if eol_date is not None and eol_date <= dates.max():
    ax.plot([eol_date], [EOL], "rX", ms=13, label=f"EOL ~{eol_date:%Y-%m}")
ax.annotate(f"{soh_at_warranty:.0f}% @ warranty", xy=(warranty_end, soh_at_warranty),
            xytext=(10, 10), textcoords="offset points", fontsize=9, color="green")
ax.set_title(f"{VIN} (Mahindra Treo) — SoH: actual vs forecast to {WARRANTY_YEARS:.0f}-yr warranty")
ax.set_ylabel("State of Health (%)"); ax.set_xlabel("Date")
ax.xaxis.set_major_locator(YearLocator()); ax.xaxis.set_major_formatter(DateFormatter("%Y"))
ax.set_ylim(70, 102); ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
Path("data/mahindra/soh").mkdir(parents=True, exist_ok=True)
out = f"data/mahindra/soh/{VIN}_forecast_to_warranty.png"
